# Sequencer

## Future Ideas
* Swing/jitter: every second note comes a bit early
* With ItsyBitsy M0
  * sync in and out (with the itsy bisty M0)
  * proper analog reference for DAC

## Timing and Intervals

### Overview
<img src="./images/timing_cv_gate.png" alt="Timing for CV and GATE" width="640"/>

The diagram illustrates the signal levels for the CV and GATE outputs for a two note sequence. 
* Interval $T = \frac{60}{BPM}$. The speed ranges from 24 to 240 BPM. Therefore $T_{min} = 0.25s$ and $T_{max} = 2.5s$
* Duty cycle $D$ ranges from 0 to 1
* Slide $S$ ranges from 0 to 1
  * This could refer to (inverse) slope instead of slide: 1/m = 0 is an abrupt transition; 1/m = 1 is the slowest constant slope (maybe 0.1V/T).
  * Could be linear or quadratic ("swoop" up/down)
* The GATE output is a logic-level output from the SAMD21, buffered/amplified to 0-5V output
* The CV output is an analog output from the SAMD21 DAC, buffered/amplified to 0-5V output
  * The DAC is 10-bit/4096 levels with a minimum conversion time of less than 3us (max 350ksps)

### Discretization
The target is <1ms resolution. The interval can be set by the ADC conversion time in free-running mode. In each interval
* sample the current level control (potentiometer)
  * ADC conversion complete
* sample the current button state (logic 0 or 1)
  * set next action if debounced
* trigger a DAC conversion with the active CV level
* write the GATE output bit
* advance the MUX address to the next inputs
* update interval counter
* calculate the next state
  * accumulate and average the level inputs ($T_{lag} = N_{avg}*8ms$) -- neccessary in addition to ADC averaging?
  * compute the next CV level (apply quantization & slide with updated levels)
  * compute the next GATE bit (duty cycle)
* did we start a new interval? button triggered or timeout
  * update lights -- 128 bytes @ 800kHz = 0.16ms, DMA compatible
  * update display -- **1024 bytes @ 400kHz = 2.56ms** (@ 800kHz = 1.28us)
* check the state of the rotary control & push button
  * menu action?

### Modes and Timing
Changes to the settings will impact the current and future state of the outputs. The following table summarizes those effects.

| Control input | Interval change? | Effect    | Update | Notes |  
|---------------|------------------|-----------|--------|-------|
| Slide         | No               | immediate | CV     | recalculate slope & intercept |
| Duty          | No               | immediate | GATE   |       |
| Quantization  | No               | next interval | CV     | apply quantization |
| Active        | No               | immediate | GATE   | |
| Piano         | No               | next interval | CV, GATE, interval ID| time quantized |
| BPM           | Yes              | immediate | all | see sketch |
| No. intervals | No               | next interval | all | if n > N, n=0 |


TODO: the following is dumb. Just keep the cursor in the same interval, same relative position.

~~In the diagram below, the impact of adjustments to the BPM are illustrated. The sequencer has a timing resolution $\Delta t$, such that each interval is composed of $N_i = \left\lfloor\frac{60}{BPM \times \Delta t}\right\rfloor = \left\lfloor\frac{60 f_{tick}}{BPM}\right\rfloor$ time steps. The index of the current time step is then $n = k_i N_i + m$, where $k_i$ is the interval index and $m$ is the time step index within the interval.~~
* hold $n$ constant, BPM adjustment changes $N_i$
* $k_i = n / N_i$
* $m = n - k_i N_i$
* $k_i = \mathrm{mod}(k_i,N)$ where $N$ is the number of intervals

![Adjusting BPM](./images/timing_adjust_speed.png)






## Controls and Menu

### Controls

* level potentiometers
  * quantization sets CV to discrete levels
  * slide changes CV in time
* step buttons
  * by mode: 
    * tap: note on/off, note enable/disable in sequence, 
    * hold: [octave -, octave +, linear, major, minor, chroma, change pattern, change output range]
  * start/pause button down: start sequence from here immediately
* start/pause button
  * tap: start/pause toggle
  * hold + step button: start from here immediately
* mode button
  * tap: change mode of step keys (note on/off, note enable/disable, ...) 
  * hold + step button: select mode immediate 
* roatary button
  * tap: menu action (in/out)
  * scroll: next item (out), adjust (in)
  
### Menu
* control: click to enter, rotate to adjust, click to exit
  * speed (BPM): 24-240
  * duty (%): 0-100
    * long press: gate on/off
  * slide (%): 0-100
  * mode: loop, bounce, random
  * quantization: linear, minor, major, chromatic
  * ? output range (V/oct 1V, V/oct 2V, ..., ?Hz/V)
* display
  * Status Bar: display menu here, underline current, click to enter -> underline number, click to exit -> back to underline label
    * BPM: heart icon + three digits
    * duty: gate icon + three digits for "gate on"; box with diag line for "gate off"
    * slide: slide icon + three digits
  * Central Widget
    * ? levels/notes
    






### ADC Timing 

The [QtPY](https://learn.adafruit.com/adafruit-qt-py/overview) is based on the [ATSAMD21E18](http://www.microchip.com/wwwproducts/en/ATSAMD21E18). In freerunning mode, a conversion is driven by a divided clock that is input to the ADC. The conversion time in cycles of the system clock is
$$
T_{conv} = N_{avg}D(6 + G + 0.5^N)2^{P+2}
$$
where $N_{avg}$ is the number of sequential conversions averaged together, $D$ is the system clock division ratio (0-255), $G$ is the conversion gain delay (1 for DIV2), $N$ is the sample length (half-cycles of the ADC clock), and $P$ is the pre-scaler factor. The ADC clock frequency is related to the system clock frequency as 

$$
f_{ADC} = \frac{f_{sys}}{D 2^{2+P}}
$$

The timing for freerunning conversion is shown below

![ADC conversion timing](./images/SAMD21E_AD_conversion_timing_single_ended.png)

When the sample length $N$ is increased, the start of the conversion is delayed accordingly. In free-running mode, this means that the sample time will overlap with the LSBs of the current conversion.

![ADC sample length](./images/SAMD21E_AD_conversion_timing_extended_sample.png)

The ADC on the SAMD21E has a 3.5pF sample capacitor and a 3.5k input source resistance. The required sample and hold time for 12-bit resolution (formula from the datasheet) is

$$
t_{samplehold} \geq 9.02(R_{sample} + R_{source})C_{sample} \to f_{ADC} \leq \frac{1}{2 t_{samplehold}}
$$

The source resistance is the sum of the ON resistance of the mux ([74HC4015](https://www.ti.com/product/CD74HC4051)) and the AC resistance of the potentiometer. From the datasheet, $R_{ON} < 200\Omega$. The worst case AC equivalent for the potentiometer will be $R_{p} = 0.25R$. 

The calculations below show parameters for the ADC configuration assuming
* 16ksps with sample time 5x larger than $t_{samplehold}$
* 100k pots (can reduce to 10k, which reduces the sample time by a factor of 10)
* The conversions will be averaged down by 8 to yield an effective sample rate of 2ksps 

The expected timing diagram is shown below

![Expected ADC timing](./images/timing_ADC_conv.png)

In [17]:
C_sample = 3.5e-12
R_sample = 3500
R_on = 200
R_full = 100000
R_p = 0.25*R_full
R_source = R_on + R_full

t_sh_min = 9.02*(R_sample + R_source)*C_sample
f_ADC_max = 0.5/t_sh_min

print(f'Min sample time is {t_sh_min*1e6:.2f}us (max ADC frequency is {f_ADC_max/1000:.1f}ksps when samplen=0)')

f_sys = 48e6
T_sys = 1.0/f_sys

D = 75
P = 0
N = 5
G = 1
resolution = 12 # bits

f_ADC = f_sys/D/2**(P+2)
T_ADC = 1.0/f_ADC
prop_delay = (resolution/2 + G) * T_ADC
sample_time = 0.5*(N + 1)*T_ADC
T_conv = prop_delay + sample_time

print(f'ADC timing')
print(f'  + f_ADC: {f_ADC/1000:.1f}kHz')
print(f'  + T_sample (samplen={N}): {sample_time*1e6}us')
print(f'  + T_conv: {T_conv*1e6:.1f}us')
print(f'  + sample rate: {0.001/T_conv:.1f}ksps')




Min sample time is 3.27us (max ADC frequency is 152.7ksps when samplen=0)
ADC timing
  + f_ADC: 160.0kHz
  + T_sample (samplen=5): 18.75us
  + T_conv: 62.5us
  + sample rate: 16.0ksps


## Music Theory

Reference: https://www.swarthmore.edu/NatSci/ceverba1/Class/e5_2006/MusicalScales.html


## Display Layout

Using [Adafruit_GFX](https://github.com/adafruit/Adafruit-GFX-Library) with the included font library. Fonts are stored in a table from which the dimensions can be deduced according to the [GFXglyph](https://github.com/adafruit/Adafruit-GFX-Library/blob/master/gfxfont.h) struct. For example, for [9pt Sans](https://github.com/adafruit/Adafruit-GFX-Library/blob/master/Fonts/FreeSans9pt7b.h)

|char | W | H | xAdv | xOff | yOff |
|-----|---|---|------|------|------|
| 0   | 8 | 13| 10 | 1 | -12 |
| 1   | 4 | 13| 10 | 3 | -12 |
| 2   | 9 | 13| 10 | 1 | -12 |
| 3   | 8 | 13| 10 | 1 | -12 |
| 4   | 7 | 13| 10 | 2 | -12 |
| 5   | 9 | 13| 10 | 1 | -12 |
| 6   | 9 | 13| 10 | 1 | -12 |
| 7   | 8 | 13| 10 | 0 | -12 |
| 8   | 9 | 13| 10 | 1 | -12 |
| 9   | 8 | 13| 10 | 1 | -12 |

So all are H=13 pixels and xAdvance is 10 pixels
